# 🧠 Deep Learning Security Models

## Advanced Neural Networks for Cybersecurity

This notebook focuses on training **deep learning models** for security classification:

- **Transformer-based Detection** - Attention mechanisms for sequence analysis
- **Convolutional Networks** - Pattern detection in security data
- **LSTM/GRU Networks** - Temporal pattern recognition
- **AutoEncoders** - Anomaly detection via reconstruction error
- **Multi-Task Learning** - Unified model for multiple security domains

In [ ]:
# Install required packages using pip magic (ensures correct kernel environment)
# Note: TensorFlow requires Python 3.9-3.11. If you see errors, switch to venv kernel or use Python 3.11

import sys
print(f'🐍 Current Python: {sys.version}')

# Check Python version
major, minor = sys.version_info[:2]
if major == 3 and 9 <= minor <= 11:
    %pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn imbalanced-learn nest_asyncio tqdm
    print('✅ All packages installed including TensorFlow')
else:
    print(f'⚠️ Python {major}.{minor} detected. TensorFlow requires Python 3.9-3.11')
    print('   Installing other packages without TensorFlow...')
    %pip install -q scikit-learn pandas numpy matplotlib seaborn imbalanced-learn nest_asyncio tqdm
    print('✅ Packages installed (without TensorFlow)')
    print('   Please switch to Python 3.9-3.11 kernel to use deep learning models')

🐍 Current Python: 3.15.0a3 (v3.15.0a3:f1eb0c0b0cd, Dec 16 2025, 08:05:19) [Clang 17.0.0 (clang-1700.6.3.2)]
⚠️ Python 3.15 detected. TensorFlow requires Python 3.9-3.11
   Installing other packages without TensorFlow...


In [ ]:
import os
import sys
import asyncio
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import json
import joblib

# ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, 
    Conv1D, MaxPooling1D, GlobalMaxPooling1D, Flatten,
    LSTM, GRU, Bidirectional, Attention, MultiHeadAttention,
    Concatenate, Add, LayerNormalization, Embedding
)
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l1_l2

from imblearn.over_sampling import SMOTE

# Config
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
np.random.seed(42)
tf.random.set_seed(42)

# Add path
sys.path.insert(0, str(Path.cwd().parent / 'app' / 'services'))

try:
    import nest_asyncio
    nest_asyncio.apply()
except:
    pass

plt.style.use('dark_background')

print('🚀 Environment ready!')
print(f'   TensorFlow: {tf.__version__}')
print(f'   GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📥 Load Security Datasets

In [ ]:
from web_security_datasets import WebSecurityDatasetManager

DATASET_DIR = Path.cwd().parent / 'datasets' / 'web_security'
manager = WebSecurityDatasetManager(str(DATASET_DIR))

# Download if needed
async def ensure_datasets():
    if len(manager.downloaded_datasets) < 5:
        print('📥 Downloading datasets...')
        await manager.download_all_datasets()
    return manager.downloaded_datasets

datasets = asyncio.run(ensure_datasets())
print(f'\n✅ {len(datasets)} datasets available')

In [ ]:
# Load combined dataset for multi-domain training
async def load_combined(max_per_ds: int = 20000):
    return await manager.get_combined_dataset(max_samples_per_dataset=max_per_ds)

combined_df = asyncio.run(load_combined())
print(f'📊 Combined dataset: {len(combined_df):,} samples')
print(f'   Features: {combined_df.shape[1]}')
print(f'   Categories: {combined_df["_category"].value_counts().to_dict()}')

## 🏗️ Deep Learning Architectures

In [ ]:
class DeepSecurityModels:
    """Advanced deep learning models for security classification."""
    
    @staticmethod
    def transformer_block(x, embed_dim, num_heads, ff_dim, dropout=0.1):
        """Transformer encoder block."""
        # Multi-head attention
        attn_output = MultiHeadAttention(
            key_dim=embed_dim, num_heads=num_heads, dropout=dropout
        )(x, x)
        x1 = LayerNormalization(epsilon=1e-6)(x + attn_output)
        
        # Feed-forward
        ff = Dense(ff_dim, activation='relu')(x1)
        ff = Dropout(dropout)(ff)
        ff = Dense(embed_dim)(ff)
        return LayerNormalization(epsilon=1e-6)(x1 + ff)
    
    @staticmethod
    def create_transformer_classifier(input_dim: int, 
                                       embed_dim: int = 64,
                                       num_heads: int = 4,
                                       ff_dim: int = 128,
                                       num_blocks: int = 2) -> Model:
        """Transformer-based security classifier."""
        inputs = Input(shape=(input_dim,))
        
        # Project to embedding dimension
        x = Dense(embed_dim)(inputs)
        x = tf.expand_dims(x, axis=1)  # Add sequence dimension
        
        # Stack transformer blocks
        for _ in range(num_blocks):
            x = DeepSecurityModels.transformer_block(x, embed_dim, num_heads, ff_dim)
        
        # Global pooling and classification
        x = tf.squeeze(x, axis=1)
        x = Dropout(0.2)(x)
        x = Dense(32, activation='relu')(x)
        outputs = Dense(1, activation='sigmoid')(x)
        
        model = Model(inputs, outputs, name='transformer_classifier')
        model.compile(
            optimizer=AdamW(learning_rate=1e-4),
            loss='binary_crossentropy',
            metrics=['accuracy', 'AUC']
        )
        return model
    
    @staticmethod
    def create_cnn_classifier(input_dim: int) -> Model:
        """1D CNN for security pattern detection."""
        inputs = Input(shape=(input_dim, 1))
        
        # Conv blocks
        x = Conv1D(64, 3, activation='relu', padding='same')(inputs)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        
        x = Conv1D(128, 3, activation='relu', padding='same')(x)
        x = BatchNormalization()(x)
        x = MaxPooling1D(2)(x)
        
        x = Conv1D(256, 3, activation='relu', padding='same')(x)
        x = GlobalMaxPooling1D()(x)
        
        # Classification head
        x = Dense(64, activation='relu')(x)
        x = Dropout(0.3)(x)
        outputs = Dense(1, activation='sigmoid')(x)
        
        model = Model(inputs, outputs, name='cnn_classifier')
        model.compile(
            optimizer=Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=['accuracy', 'AUC']
        )
        return model
    
    @staticmethod
    def create_lstm_classifier(input_dim: int) -> Model:
        """Bidirectional LSTM for sequence analysis."""
        inputs = Input(shape=(input_dim, 1))
        
        x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
        x = Dropout(0.3)(x)
        x = Bidirectional(LSTM(32))(x)
        x = Dropout(0.3)(x)
        
        x = Dense(32, activation='relu')(x)
        outputs = Dense(1, activation='sigmoid')(x)
        
        model = Model(inputs, outputs, name='lstm_classifier')
        model.compile(
            optimizer=Adam(learning_rate=1e-3),
            loss='binary_crossentropy',
            metrics=['accuracy', 'AUC']
        )
        return model
    
    @staticmethod
    def create_autoencoder(input_dim: int, encoding_dim: int = 32) -> tuple:
        """Autoencoder for anomaly detection."""
        # Encoder
        inputs = Input(shape=(input_dim,))
        x = Dense(128, activation='relu')(inputs)
        x = BatchNormalization()(x)
        x = Dense(64, activation='relu')(x)
        x = BatchNormalization()(x)
        encoded = Dense(encoding_dim, activation='relu', name='encoding')(x)
        
        # Decoder
        x = Dense(64, activation='relu')(encoded)
        x = BatchNormalization()(x)
        x = Dense(128, activation='relu')(x)
        x = BatchNormalization()(x)
        decoded = Dense(input_dim, activation='linear')(x)
        
        autoencoder = Model(inputs, decoded, name='autoencoder')
        autoencoder.compile(optimizer=Adam(1e-3), loss='mse')
        
        encoder = Model(inputs, encoded, name='encoder')
        
        return autoencoder, encoder
    
    @staticmethod
    def create_multi_task_model(input_dim: int, num_tasks: int = 3) -> Model:
        """Multi-task model for multiple security domains."""
        inputs = Input(shape=(input_dim,))
        
        # Shared layers
        shared = Dense(256, activation='relu')(inputs)
        shared = BatchNormalization()(shared)
        shared = Dropout(0.3)(shared)
        shared = Dense(128, activation='relu')(shared)
        shared = BatchNormalization()(shared)
        shared = Dropout(0.2)(shared)
        shared = Dense(64, activation='relu')(shared)
        
        # Task-specific heads
        outputs = []
        task_names = ['phishing', 'malware', 'intrusion']
        for i in range(min(num_tasks, len(task_names))):
            task_layer = Dense(32, activation='relu', name=f'{task_names[i]}_hidden')(shared)
            task_output = Dense(1, activation='sigmoid', name=f'{task_names[i]}_output')(task_layer)
            outputs.append(task_output)
        
        model = Model(inputs, outputs, name='multi_task_security')
        model.compile(
            optimizer=Adam(1e-3),
            loss={f'{task_names[i]}_output': 'binary_crossentropy' for i in range(len(outputs))},
            metrics=['accuracy']
        )
        return model

print('✅ Deep learning architectures defined')

## 🎯 Training Pipeline

In [ ]:
def prepare_data_for_training(df: pd.DataFrame, max_features: int = 50) -> tuple:
    """Prepare data for deep learning training."""
    
    # Find target column
    target_candidates = ['is_malicious', 'is_attack', 'is_malware', 'is_spam', 
                        'is_dga', 'is_miner', 'label', 'result']
    target_col = None
    for col in target_candidates:
        if col in df.columns:
            target_col = col
            break
    
    if target_col is None:
        # Find binary column
        for col in df.columns:
            if df[col].nunique() == 2 and col not in ['_category', '_dataset_id']:
                target_col = col
                break
    
    if target_col is None:
        raise ValueError('No target column found')
    
    # Select numeric features
    exclude = [target_col, '_category', '_dataset_id', 'source_dataset', 'url', 'payload', 'domain']
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
    
    # Limit features
    if len(feature_cols) > max_features:
        feature_cols = feature_cols[:max_features]
    
    X = df[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)
    y = df[target_col].astype(int)
    
    # Scale
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, y.values, feature_cols, scaler

# Prepare data
X, y, features, scaler = prepare_data_for_training(combined_df)
print(f'📊 Data prepared: {X.shape}')
print(f'   Features: {len(features)}')
print(f'   Class balance: {np.bincount(y)}')

In [ ]:
# Split and balance data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Balance training data
try:
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    print(f'✅ After SMOTE: {len(X_train_balanced):,} training samples')
except:
    X_train_balanced, y_train_balanced = X_train, y_train
    print('⚠️ SMOTE skipped')

print(f'   Train: {len(X_train_balanced):,} | Test: {len(X_test):,}')

In [ ]:
# Training callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
]

# Train Transformer model
print('🔄 Training Transformer model...')
transformer = DeepSecurityModels.create_transformer_classifier(X.shape[1])

history_transformer = transformer.fit(
    X_train_balanced, y_train_balanced,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

transformer_pred = (transformer.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
transformer_auc = roc_auc_score(y_test, transformer.predict(X_test, verbose=0))
print(f'\n✅ Transformer AUC: {transformer_auc:.4f}')

In [ ]:
# Train CNN model
print('🔄 Training CNN model...')

X_train_cnn = X_train_balanced.reshape(-1, X_train_balanced.shape[1], 1)
X_test_cnn = X_test.reshape(-1, X_test.shape[1], 1)

cnn = DeepSecurityModels.create_cnn_classifier(X.shape[1])

history_cnn = cnn.fit(
    X_train_cnn, y_train_balanced,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

cnn_pred = (cnn.predict(X_test_cnn, verbose=0) > 0.5).astype(int).flatten()
cnn_auc = roc_auc_score(y_test, cnn.predict(X_test_cnn, verbose=0))
print(f'\n✅ CNN AUC: {cnn_auc:.4f}')

In [ ]:
# Train LSTM model
print('🔄 Training LSTM model...')

lstm = DeepSecurityModels.create_lstm_classifier(X.shape[1])

history_lstm = lstm.fit(
    X_train_cnn, y_train_balanced,  # Same shape as CNN
    validation_split=0.2,
    epochs=30,  # LSTM is slower
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

lstm_pred = (lstm.predict(X_test_cnn, verbose=0) > 0.5).astype(int).flatten()
lstm_auc = roc_auc_score(y_test, lstm.predict(X_test_cnn, verbose=0))
print(f'\n✅ LSTM AUC: {lstm_auc:.4f}')

In [ ]:
# Train Autoencoder for anomaly detection
print('🔄 Training Autoencoder...')

# Train only on normal samples
X_normal = X_train_balanced[y_train_balanced == 0]

autoencoder, encoder = DeepSecurityModels.create_autoencoder(X.shape[1])

history_ae = autoencoder.fit(
    X_normal, X_normal,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# Anomaly scores based on reconstruction error
reconstructions = autoencoder.predict(X_test, verbose=0)
mse = np.mean(np.power(X_test - reconstructions, 2), axis=1)
threshold = np.percentile(mse, 90)  # Top 10% as anomalies
ae_pred = (mse > threshold).astype(int)
ae_auc = roc_auc_score(y_test, mse)
print(f'\n✅ Autoencoder AUC: {ae_auc:.4f}')

## 📊 Model Comparison

In [ ]:
# Compare all models
results = {
    'Transformer': {'pred': transformer_pred, 'auc': transformer_auc},
    'CNN': {'pred': cnn_pred, 'auc': cnn_auc},
    'LSTM': {'pred': lstm_pred, 'auc': lstm_auc},
    'Autoencoder': {'pred': ae_pred, 'auc': ae_auc}
}

# Results table
print('📊 Deep Learning Model Comparison')
print('=' * 60)
print(f'{"Model":<15} {"Accuracy":<12} {"F1":<12} {"AUC":<12}')
print('-' * 60)

for name, res in results.items():
    acc = accuracy_score(y_test, res['pred'])
    f1 = f1_score(y_test, res['pred'])
    print(f'{name:<15} {acc:<12.4f} {f1:<12.4f} {res["auc"]:<12.4f}')

# Best model
best_model = max(results.items(), key=lambda x: x[1]['auc'])
print(f'\n🏆 Best Model: {best_model[0]} (AUC: {best_model[1]["auc"]:.4f})')

In [ ]:
# Visualize ROC curves
plt.figure(figsize=(10, 8))

# Get probabilities
probs = {
    'Transformer': transformer.predict(X_test, verbose=0).flatten(),
    'CNN': cnn.predict(X_test_cnn, verbose=0).flatten(),
    'LSTM': lstm.predict(X_test_cnn, verbose=0).flatten(),
    'Autoencoder': mse / mse.max()  # Normalized MSE
}

colors = ['#4ecdc4', '#ff6b6b', '#ffe66d', '#95e1d3']
for (name, prob), color in zip(probs.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = results[name]['auc']
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('🎯 Deep Learning ROC Comparison', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Training history visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

histories = [
    ('Transformer', history_transformer),
    ('CNN', history_cnn),
    ('LSTM', history_lstm)
]

for ax, (name, hist) in zip(axes, histories):
    ax.plot(hist.history['loss'], label='Train Loss')
    ax.plot(hist.history['val_loss'], label='Val Loss')
    ax.set_title(f'{name} Training', color='white')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 💾 Save Models

In [ ]:
# Save trained models
MODELS_DIR = Path.cwd().parent / 'models' / 'deep_learning'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('💾 Saving models...')

# Save Keras models
transformer.save(MODELS_DIR / 'transformer_security.keras')
cnn.save(MODELS_DIR / 'cnn_security.keras')
lstm.save(MODELS_DIR / 'lstm_security.keras')
autoencoder.save(MODELS_DIR / 'autoencoder_security.keras')
encoder.save(MODELS_DIR / 'encoder_security.keras')

# Save scaler and config
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
joblib.dump(features, MODELS_DIR / 'feature_names.pkl')

# Save metrics
metrics = {
    name: {'accuracy': float(accuracy_score(y_test, r['pred'])),
           'f1': float(f1_score(y_test, r['pred'])),
           'auc': float(r['auc'])}
    for name, r in results.items()
}
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n✅ Models saved to {MODELS_DIR}')

## 🎉 Summary

### Trained Models:
- **Transformer** - Attention-based classifier
- **CNN** - Convolutional pattern detector
- **LSTM** - Sequence analyzer
- **Autoencoder** - Anomaly detector

### Output Files:
```
models/deep_learning/
├── transformer_security.keras
├── cnn_security.keras
├── lstm_security.keras
├── autoencoder_security.keras
├── encoder_security.keras
├── scaler.pkl
├── feature_names.pkl
└── metrics.json
```

These models are ready for integration with the Agentic AI security system!